# Kho-Sight — Train the role detector on Google Colab

Fine-tunes a YOLO model on your annotated Kho-Kho frames so the pipeline reliably
detects **chaser-seated / chaser-active / runner / cone**. Free Colab T4 GPU is enough.

**Before opening this notebook** (done once, on any laptop — no GPU needed):
1. Extract frames from your match videos: `python scripts/prepare_dataset.py frames --videos matches/ --out frames/` (or use Step 2b below to do it in Colab from Google Drive).
2. (Optional but recommended) Pre-label them: `python scripts/auto_label.py --frames frames/ --out labels/` — or run it here in Colab where the big model is fast.
3. Annotate/correct in [Roboflow](https://roboflow.com) (free tier): create an **Object Detection** project with classes exactly `chaser-seated, chaser-active, runner, cone`, upload frames (+ labels), correct boxes, then *Generate a dataset version* and export as **YOLOv11**.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# Step 0 — confirm the GPU is on
!nvidia-smi

In [ ]:
# Step 1 — install
%pip -q install ultralytics roboflow

## Step 2 — get your dataset into Colab
Pick ONE of the options below.

In [ ]:
# Option A (recommended): pull the dataset straight from Roboflow.
# In Roboflow: your project -> Versions -> Export -> YOLOv11 -> 'show download code'
# and paste the values here.
from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("yolov11")
DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)

In [ ]:
# Option B: dataset zip on Google Drive (a Roboflow/CVAT YOLO export you zipped).
# Expected inside the zip: data.yaml, train/images, train/labels, valid/images, valid/labels
from google.colab import drive
drive.mount('/content/drive')
!unzip -qo /content/drive/MyDrive/khosight/dataset.zip -d /content/dataset
DATA_YAML = "/content/dataset/data.yaml"
!head -20 $DATA_YAML

In [ ]:
# Optional Step 2b — no dataset yet? Mine frames + pre-labels here in Colab.
# Upload a few match videos to Drive (e.g. MyDrive/khosight/videos/) first.
# Then download the frames+labels zip this produces, import it into Roboflow,
# correct the labels, and come back to Step 2.
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/HardhikTottempudi/Kho-sight /content/khosight_repo
!python /content/khosight_repo/scripts/prepare_dataset.py frames \
    --videos /content/drive/MyDrive/khosight/videos --out /content/frames --every-s 2.0
!python /content/khosight_repo/scripts/auto_label.py \
    --frames /content/frames --out /content/prelabels --model yolo11x.pt
!cd /content && zip -qr frames_prelabels.zip frames prelabels
from google.colab import files; files.download('/content/frames_prelabels.zip')

## Step 3 — train
~1.5–3 h on a T4 for ~2–3k images at these settings. If Colab disconnects,
re-run with `resume=True` pointing at the last run.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")   # yolo11m.pt = a bit more accurate, ~2x slower
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=1280,     # seated chasers are small at court distance; drop to 960 if slow
    batch=-1,       # auto-fit to GPU memory
    patience=25,    # early stop when validation stops improving
    degrees=5, scale=0.3, mosaic=0.5,   # static court cameras -> mild augmentation
    project="runs/khosight", name="roles",
)

## Step 4 — check quality (aim for mAP50 ≥ 0.85 on every class, especially chaser-seated)

In [ ]:
metrics = model.val()
print(metrics)

# eyeball a few validation predictions
import glob
from IPython.display import Image, display
val_imgs = glob.glob(dataset.location + "/valid/images/*")[:3] if 'dataset' in dir() \
           else glob.glob("/content/dataset/valid/images/*")[:3]
for p in model.predict(val_imgs, conf=0.35, save=True):
    pass
for p in glob.glob("runs/detect/predict*/*")[:3]:
    display(Image(p, width=900))

## Step 5 — save the weights (Drive + local download)

In [ ]:
import glob
from google.colab import drive, files

drive.mount('/content/drive')
# ultralytics versions differ in where they nest save_dir; find best.pt robustly
best = sorted(glob.glob('/content/**/weights/best.pt', recursive=True))[-1]
print('weights:', best)
!mkdir -p /content/drive/MyDrive/khosight
!cp "{best}" /content/drive/MyDrive/khosight/khosight_roles_best.pt
files.download(best)

## Step 6 — use it on your laptop (CPU is fine for offline analysis)
```bash
python -m khosight analyze --video match.mp4 --calib court_calib.json \
    --team-a Lions --team-b Tigers --half Lions:0:180 --half Tigers:240:420 \
    --role-model khosight_roles_best.pt --out report
```
The fine-tuned detector now supplies team/role labels (replacing jersey-colour
clustering) while the pretrained pose model keeps supplying keypoints.

**Iterate:** run analysis, find frames where it errs, add those frames to Roboflow,
retrain. 2–3 rounds of this active-learning loop is usually what gets match-grade accuracy.